# Fine‑Tuning BERT‑style Models on a Small Corpus

> **Context**  
> We are working with **≈ 7 000 labelled texts**, which firmly places us in a *low‑data* regime.  
> In such settings, empirical and theoretical work [^1] shows that **larger pretrained models often compensate for data scarcity**, but only when they are tuned carefully.

---

## Why we start small 🪶

| Challenge | Consequence with large models | Our mitigation |
|-----------|--------------------------------|----------------|
| **Longer iteration time** (more parameters ⟶ slower epochs) | Exhaustive hyper‑parameter search becomes impractical. | **Begin with the tiniest BERT variants** (4 – 18 M params) so we can sweep many learning‑rates & batch‑sizes quickly. |
| **Hyper‑parameter sensitivity** | A poor LR / batch‑size combo wastes compute and may stall training. | **Grid‑search on the small models** to discover stable regions before scaling up. |

---

## Step‑wise scaling strategy 🚀

1. **Tune on the tiny model family**  
   * Grid over `batch_size ∈ {8, 16, 32, 64}` × `lr ∈ {3 e‑5 … 3 e‑4}`.  
   * Record the best‐performing configuration.

2. **Scale the hidden size & depth**  
   * For each larger model:  
     * **Halve the learning‑rate** when the parameter count roughly doubles.  
     * **Optionally double the batch‑size** for better gradient‑noise balance.

3. **Local (not global) search** around the inherited HPs  
   * ± 1 × LR factor, ± 1 batch‑size step.  
   * Early‑stop poorly behaving runs to save quota.

This *progressive widening* lets us exploit the robustness insights gained
from small models while still converging on good HPs for the bigger ones.

---

## Notebook roadmap 🗺️

1. **Data loading & label encoding**  
2. **Model zoo definition** (Google’s Tiny → Base BERT checkpoints)  
3. **Training**: loss weighting, logging, history capture  
6. **Results table & visual analytics**  
   * accuracy / loss curves  
   * param‑count vs accuracy plots  
   * batch‑size & LR heat‑maps  

---

[^1]: e.g. Turc et al., *Well‑Read Students Learn Better: On the Importance of Pre‑training Compact Models*, 2019 [(link)](https://arxiv.org/abs/1908.08962).


In [ ]:
# @ Import libs
import torch, random, numpy as np, os, time, logging, math, json, collections, re
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timezone, MINYEAR
import datetime
from tqdm.auto import tqdm
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoModelForSequenceClassification,
                          AutoTokenizer, 
                          get_linear_schedule_with_warmup)

In [ ]:
# @title Helper functions

def set_seed(seed=None, seed_torch=True):
  """
  Function that controls randomness. NumPy and random modules must be imported.

  Args:
    seed : Integer
      A non-negative integer that defines the random state. Default is `None`.
    seed_torch : Boolean
      If `True` sets the random seed for pytorch tensors, so pytorch module
      must be imported. Default is `True`.

  Returns:
    Nothing.
  """
  if seed is None:
    seed = np.random.choice(2 ** 32)
  random.seed(seed)
  np.random.seed(seed)
  if seed_torch:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

  print(f'Random seed {seed} has been set.')

# In case that `DataLoader` is used
def seed_worker(worker_id):
  """
  DataLoader will reseed workers following randomness in
  multi-process data loading algorithm.

  Args:
    worker_id: integer
      ID of subprocess to seed. 0 means that
      the data will be loaded in the main process
      Refer: https://pytorch.org/docs/stable/data.html#data-loading-randomness for more details

  Returns:
    Nothing
  """
  worker_seed = torch.initial_seed() % 2**32
  np.random.seed(worker_seed)
  random.seed(worker_seed)

def set_device():
  """
  Set the device. CUDA if available, CPU otherwise

  Args:
    None

  Returns:
    Nothing
  """
  device = "cuda" if torch.cuda.is_available() else "cpu"
  if device != "cuda":
    print("WARNING: For this notebook to perform best, "
        "if possible, in the menu under `Runtime` -> "
        "`Change runtime type.`  select `GPU` ")
  else:
    print("GPU is enabled in this notebook.")

  return device

def save_history(tag, history, results_dir="results"):
    """
    Write `history` to results/{tag}_{YYYYmmddTHHMMSS}_history.json.
    """
    Path(results_dir).mkdir(exist_ok=True)
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S")   # e.g. 20250718T182045
    dst   = Path(results_dir) / f"{tag}_{stamp}_history.json"
    with open(dst, "w") as f:
        json.dump(history, f, indent=2)
    print(f"✓ saved → {dst}")

def _ts_or_min(ts):
    """Return a comparable datetime (MINYEAR) when ts is None."""
    return ts or datetime.datetime(MINYEAR, 1, 1, tzinfo=timezone.utc)

_PARAM_TABLE = {
    2:  {128: 4.4, 256:  9.7, 512: 22.8, 768:  39.2},
    4:  {128: 4.8, 256: 11.3, 512: 29.1, 768:  53.4},
    6:  {128: 5.2, 256: 12.8, 512: 35.4, 768:  67.5},
    8:  {128: 5.6, 256: 14.4, 512: 41.7, 768:  81.7},
    10: {128: 6.0, 256: 16.0, 512: 48.0, 768:  95.9},
    12: {128: 6.4, 256: 17.6, 512: 54.3, 768: 110.1},
}
_LH_RE   = re.compile(r"L-(\d+)_H-(\d+)_A-")            # captures L, H
_BSLR_RE = re.compile(r"_bs(\d+)_lr([0-9.+eE-]+)")      # captures bs, lr

def _mean_last_k(seq, k=5):
    """
    Return the mean of the last k elements of seq (list or np.ndarray).
    If seq has fewer than k items, use all of them.
    """
    if not seq:                       # empty list guard
        return None
    tail = seq[-k:]                   # slice works even if len(seq) < k
    return float(sum(tail)) / len(tail)

def _model_params(L, H):
    """Return model size (millions of params) given L & H, or None if absent."""
    return _PARAM_TABLE.get(L, {}).get(H)

def _ts_or_min(ts):
    """Return a comparable datetime (MINYEAR) when ts is None."""
    return ts or datetime.datetime(MINYEAR, 1, 1, tzinfo=timezone.utc)

def _extract_meta(tag):
    """Return (L, H, bs, lr) as ints/floats or None."""
    L = H = bs = lr = None
    if (m := _LH_RE.search(tag)):
        L, H = int(m.group(1)), int(m.group(2))
    if (m := _BSLR_RE.search(tag)):
        bs, lr = int(m.group(1)), float(m.group(2))
    return L, H, bs, lr

def results_table(histories, class_labels=None, k_tail=5):
    """
    histories : dict  {tag : history_dict}
    class_labels : list[str] or None
        Labels for the classes in the same order as f1_each.
        If None → ['cls0', 'cls1', …]
    k_tail : how many trailing points to average for *_m{k_tail} columns
    """
    rows = []
    for tag, hist in histories.items():
        L, H, bs, lr = _extract_meta(tag)

        # scalar test metrics ------------------------------------------------
        acc      = hist["test_acc"][0]  if isinstance(hist["test_acc"],  list) else hist["test_acc"]
        f1_each  = hist["f1_each"][0]   if isinstance(hist["f1_each"],  list) else hist["f1_each"]

        if class_labels is None:
            class_labels = [f"cls{i}" for i in range(len(f1_each))]

        row = {"tag": tag,
               "L": L, "H": H,
               "bs": bs, "lr": lr,
               "acc": acc,
               "params_M": _model_params(L, H)}

        # per‑class F1s ------------------------------------------------------
        for lbl, f1 in zip(class_labels, f1_each):
            row[f"F1_{lbl}"] = f1

        # rolling mean of last k_tail points ---------------------------------
        for metric, values in hist.items():
            # keep only time‑series metrics (lists of scalars)
            if metric in {"pct_epoch"}:
                continue
            if isinstance(values, list) and values and isinstance(values[-1], (int, float)):
                row[f"{metric}_m{k_tail}"] = _mean_last_k(values, k=k_tail)

        rows.append(row)

    df = pd.DataFrame(rows)
    df = df.sort_values(["L", "H", "bs", "lr"]).reset_index(drop=True)
    return df

def load_histories(results_dir="results",
                   keep="latest",
                   L=None,
                   H=None,
                   bs=None,
                   lr=None):
    """
    Same filters as before, but dict is *sorted numerically* by
    (L, H, bs, lr, tag) before returning.
    """
    L_set  = set(L)  if L  is not None else None
    H_set  = set(H)  if H  is not None else None
    bs_set = set(bs) if bs is not None else None
    lr_set = {float(x) for x in lr} if lr is not None else None

    tmp = {}   # tag → (hist, ts)

    for path in Path(results_dir).glob("*_history.json"):
        stem = path.stem[:-8] if path.stem.endswith("_history") else path.stem
        try:
            tag, ts_str = stem.rsplit("_", 1)
            ts = datetime.datetime.strptime(ts_str, "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)
        except ValueError:
            tag, ts = stem, None

        Lv, Hv, bsv, lrv = _extract_meta(tag)

        # ── apply filters ─────────────────────────────────────
        if (L_set and (Lv not in L_set)) or \
           (H_set and (Hv not in H_set)) or \
           (bs_set and (bsv not in bs_set)) or \
           (lr_set and (lrv not in lr_set)):
            continue

        # ── load JSON ────────────────────────────────────────
        with path.open() as f:
            data = json.load(f)

        if keep == "all":
            tmp.setdefault(tag, []).append((data, ts, Lv, Hv, bsv, lrv))
        else:  # keep newest per tag
            if tag not in tmp or _ts_or_min(ts) > _ts_or_min(tmp[tag][1]):
                tmp[tag] = (data, ts, Lv, Hv, bsv, lrv)

    # ── flatten + sort numerically ───────────────────────────
    def sort_key(item):
        tag, payload = item
        hist, ts, Lv, Hv, bsv, lrv = payload if keep != "all" else payload[0]
        return (Lv or 1e9, Hv or 1e9, bsv or 1e9, lrv or 1e9, tag)

    ordered_pairs = sorted(tmp.items(), key=sort_key)

    # ── strip timestamps & meta before return ───────────────
    if keep == "all":
        histories = {tag: [h for h, *_ in lst] for tag, lst in ordered_pairs}
    else:
        histories = {tag: hist for tag, (hist, *_ ) in ordered_pairs}

    print(f"✓ loaded {len(histories)} run(s) after filtering")
    return histories

def save_timelog(timelog, fname="results/timing_log.json"):
    """
    Overwrites/creates a JSON file with the current timelog list.
    """
    Path(fname).parent.mkdir(exist_ok=True)
    with open(fname, "w") as f:
        json.dump(timelog, f, indent=2)
    print(f"✓ timing log updated → {fname}")

def plot_history_percent(histories, metric_key, title):
    plt.figure(figsize=(7,4))
    for tag, hist in histories.items():
        plt.plot(hist["pct_epoch"], hist[metric_key], label=tag)
    plt.xlabel("% of total epochs")
    plt.ylabel(metric_key.replace("_"," "))
    plt.title(title)
    plt.legend()
    plt.grid(); plt.tight_layout()

def plot_all_history_percent(histories,
                         metric_key=["tr_loss", "val_loss", "tr_acc", "val_acc"],
                         title=["Training loss", "Validation loss",
                                "Training accuracy", "Validation accuracy"]):
    for i in range(len(metric_key)):
        plot_history_percent(histories, metric_key[i],  title[i])

def plot_heatmap(
    df: pd.DataFrame,
    index: str = "bs",
    columns: str = "lr",
    values: str = "acc",
    *,
    aggfunc: str = "mean",
    title: str = None,
    cmap: str = "viridis",
    figsize: tuple[int, int] = (6, 4),
    fmt_xtick: str = "{:.0g}",
    fmt_ytick: str = "{:.0g}",
):
    """
    Draw a heat-map of `values` over `index` x `columns`.

    Parameters
    ----------
    df        : DataFrame with the data to plot
    index     : column for heat-map rows   (e.g. "bs")
    columns   : column for heat-map cols   (e.g. "lr")
    values    : metric to visualise (e.g. "acc", "val_loss_m5")
    aggfunc   : how to combine duplicates (string or np function)
    title     : plot title; if None, auto-generated
    cmap      : matplotlib colour-map
    figsize   : figure size
    fmt_xtick : format string or function for x-tick labels
    fmt_ytick : same for y-tick labels

    Returns
    -------
    ax : matplotlib Axes (so you can further tweak or save)
    """
    # ① pivot – average duplicates automatically
    pivot = (df.pivot_table(index=index,
                            columns=columns,
                            values=values,
                            aggfunc=aggfunc)
               .sort_index()
               .sort_index(axis=1))

    # ② plot
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(pivot, aspect="auto", cmap=cmap)

    # ③ tick labels
    ax.set_xticks(np.arange(len(pivot.columns)))
    xticks = pivot.columns
    ax.set_xticklabels([fmt_xtick(c) if callable(fmt_xtick) else fmt_xtick.format(c)
                        for c in xticks])

    ax.set_yticks(np.arange(len(pivot.index)))
    yticks = pivot.index
    ax.set_yticklabels([fmt_ytick(r) if callable(fmt_ytick) else fmt_ytick.format(r)
                        for r in yticks])

    ax.set_xlabel(columns)
    ax.set_ylabel(index)

    if title is None:
        title = f"{values} heat-map ({index} × {columns})"
    ax.set_title(title)

    plt.colorbar(im, ax=ax, label=values)
    plt.tight_layout()
    return ax

def plot_metrics_vs_x(df, x_key, metric_cols, title,
                      deg=1, figsize=(6,4), alpha=0.75):
    """
    df          : pd.DataFrame
    x_key       : column for x‑axis (log‑scaled)
    metric_cols : dict {legend_label : column_name_in_df}
    title       : plot title
    deg         : poly degree for trend (1 = straight line in log‑space)
    """
    x_raw = df[x_key].to_numpy()
    logx  = np.log10(x_raw)          # use log10 so slope ≈ “per decade”

    plt.figure(figsize=figsize)
    plt.xscale("log")                # ① log X‑axis

    for lbl, col in metric_cols.items():
        if col not in df.columns:
            continue
        y = df[col].to_numpy()

        # scatter
        plt.scatter(x_raw, y, label=lbl, alpha=alpha)

        # fit & draw trend
        if len(x_raw) >= deg + 1:
            coeffs = np.polyfit(logx, y, deg)
            xs_log = np.linspace(logx.min(), logx.max(), 200)
            xs_lin = 10**xs_log                         # back to linear scale
            ys_fit = np.polyval(coeffs, xs_log)
            plt.plot(xs_lin, ys_fit)

    plt.xlabel(x_key)
    plt.ylabel("Score")
    plt.title(title)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()
    plt.show()

SEED = 2025
set_seed(seed=SEED)
device = set_device()

In [ ]:
# @title Load & preprocess Hippocorpus dataset

url = 'https://raw.githubusercontent.com/pinchunc/NMA_DL_SentimentAnalysis/main/data/hippoCorpusV2.csv'
df  = pd.read_csv(url)

STRESS = True
if STRESS:
    # map stress scores → binary “Low / High”
    map_ = {1:"Low", 2:"-", 3:"High", 4:"High", 5:"High"}
    df["stressful"] = df["stressful"].map(map_)
    df = df[df["stressful"] != "-"]
    df["metric"] = df["stressful"]
else:
    df["metric"] = df["memType"]

# encode labels
lbl2id = {lbl: i for i, lbl in enumerate(sorted(df["metric"].unique()))}
id2lbl = {v: k for k, v in lbl2id.items()}
df["label_id"] = df["metric"].map(lbl2id)

print("Samples per class:")
print(df["metric"].value_counts(), "\n")
print("Label mapping:", lbl2id)


In [ ]:
# @title 2 ️Train/val/test split
from sklearn.model_selection import train_test_split

train_df, tmp_df = train_test_split(df, test_size=0.2, stratify=df["label_id"], random_state=42)
val_df, test_df  = train_test_split(tmp_df, test_size=0.5, stratify=tmp_df["label_id"], random_state=42)

print(f"Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")


In [ ]:
# @title 3 Define HF model catalogue
# ——— 24 compact BERTs ———
def google_small_name(L, H):      # A = H//64 in Google’s naming scheme
    return f"google/bert_uncased_L-{L}_H-{H}_A-{H//64}"

tiny_name = google_small_name(2,128)                       # BERT‑tiny
tiny_deep_name = google_small_name(4,128)                  # BERT‑tiny
mini_name = google_small_name(4,256)                       # BERT‑mini
mini_deep_name = google_small_name(8,256)                  # BERT‑mini
medium_name = google_small_name(8,512)                     # BERT‑medium
base_name = google_small_name(12,768)                      # BERT‑base
# robert_name = "roberta-base"                             # RoBERTa‑base

model_zoo = [tiny_name, tiny_deep_name, mini_name, mini_deep_name, medium_name, base_name]

print(model_zoo)


In [ ]:
# @title 4 Dataset class & helpers
class HipocorpusDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.texts = df["story"].tolist()
        self.labels = df["label_id"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], 
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

def build_loaders(tokenizer, bs):
    return (DataLoader(HipocorpusDataset(train_df, tokenizer), batch_size=bs, shuffle=True, drop_last=False),
            DataLoader(HipocorpusDataset(val_df,   tokenizer), batch_size=bs, shuffle=False),
            DataLoader(HipocorpusDataset(test_df,  tokenizer), batch_size=bs, shuffle=False))


In [ ]:
# @title 5 Training loop
def run_experiment(model_name,
                   batch_size=32,
                   lr=3e-5,
                   num_epochs=4,
                   log_every=None):
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSequenceClassification.from_pretrained(
                    model_name,
                    num_labels=len(lbl2id)).to(device)

    train_loader, val_loader, test_loader = build_loaders(tokenizer, batch_size)

    # weight balance in the loss function
    label_list  = [train_loader.dataset[i]["labels"]
                   for i in range(len(train_loader.dataset))]
    label_vec   = torch.tensor(label_list, dtype=torch.long)
    counts      = torch.bincount(label_vec,
                                 minlength=len(lbl2id))    # per‑class counts
    weights     = (1.0 / counts.float()).to(device)        # inverse‑freq weights
    loss_fct    = torch.nn.CrossEntropyLoss(weight=weights)

    
    if log_every is None:
        log_every = max(1, math.ceil(len(train_loader) / 10))

    optim = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader)*num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optim, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    history = collections.defaultdict(list)   # auto‑creates keys

    global_step = 0
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, seen = 0,0,0

        for step, batch in enumerate(train_loader, start=1):
            batch = {k:v.to(device) for k,v in batch.items()}
            outs  = model(**batch)
            loss  = loss_fct(outs.logits, batch["labels"])
            loss.backward()
            optim.step(); scheduler.step(); optim.zero_grad()

            running_loss += loss.item()*batch["labels"].size(0)
            preds = outs.logits.argmax(dim=-1)
            correct += (preds==batch["labels"]).sum().item()
            seen    += batch["labels"].size(0)
            global_step += 1

            if step % log_every==0 or step==len(train_loader):
                # ─── validation ───
                model.eval()
                with torch.no_grad():
                    v_loss, v_correct, v_seen = 0,0,0
                    for vb in val_loader:
                        vb  = {k:v.to(device) for k,v in vb.items()}
                        v_o = model(**vb)
                        v_loss += v_o.loss.item()*vb["labels"].size(0)
                        vp = v_o.logits.argmax(dim=-1)
                        v_correct += (vp==vb["labels"]).sum().item()
                        v_seen += vb["labels"].size(0)

                tr_loss = running_loss/seen
                tr_acc  = correct/seen
                val_loss= v_loss/v_seen
                val_acc = v_correct/v_seen
                pct     = 100*((epoch + step/len(train_loader))/num_epochs)

                print(f"[{model_name} | {pct:5.1f}%] "
                      f"tr_loss {tr_loss:.4f} val_loss {val_loss:.4f} "
                      f"tr_acc {tr_acc:.3f} val_acc {val_acc:.3f}")

                # store to history
                for k,v in [("pct_epoch", pct),
                            ("tr_loss", tr_loss), ("val_loss", val_loss),
                            ("tr_acc", tr_acc),   ("val_acc", val_acc)]:
                    history[k].append(v)
                model.train()

    # ─── final test ───
    model.eval(); y_true, y_pred = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k:v.to(device) for k,v in batch.items()}
            o = model(**batch)
            y_true.extend(batch["labels"].cpu().tolist())
            y_pred.extend(o.logits.argmax(dim=-1).cpu().tolist())

    test_acc  = accuracy_score(y_true, y_pred)
    f1_micro  = f1_score(y_true, y_pred, average='micro')
    f1_each   = f1_score(y_true, y_pred, average=None,
                         labels=list(range(len(lbl2id)))).tolist()

    print("\n===== FINAL TEST RESULTS =====")
    print("Accuracy :", test_acc)
    print("F1 micro :", f1_micro)
    for i,f in enumerate(f1_each):
        print(f"F1 class {id2lbl[i]} :", f)

    # save test metrics in history too
    history["test_acc"].append(test_acc)
    history["f1_micro"].append(f1_micro)
    history["f1_each"].append(f1_each)

    return dict(history)


In [ ]:
# @title 6 Grid search driver (small demo)
model_zoo = [tiny_name, tiny_deep_name, mini_name, mini_deep_name, medium_name]

# !! The FineTuning is done, see the next cells on training history
configs = [
    # (tiny_name, 32, 1e-4),
    # (tiny_deep_name, 32, 1e-4),
    # (mini_name, 32, 1e-4),
    # (mini_name, 64, 1e-4),
    # (mini_deep_name, 32, 1e-4),
    # (mini_deep_name, 32, 5e-5),
    # (medium_name, 64, 5e-5),
    # (medium_name, 128, 1e-4),
    # (base_name, 128, 5e-5),
]

histories = {}
timelog   = []
for m,bs,lr in configs:
    tag = f"{Path(m).name}_bs{bs}_lr{lr}"
    print("\n" + "="*20, tag, "="*20)
    start = time.time()
    histories[tag] = run_experiment(m, batch_size=bs, lr=lr,
                                        num_epochs=5, log_every=None)
    elapsed = time.time() - start
    save_history(tag, histories[tag])

    # append timing info and persist
    timelog.append({"tag": tag,
                    "model": Path(m).name,
                    "bs": bs,
                    "lr": lr,
                    "seconds": elapsed})
    save_timelog(timelog) 


In [ ]:
histories_stress_bs_test = load_histories(results_dir="results/bert_finetuning/results_stress/bs_test")
histories_stress_lr_test = load_histories(results_dir="results/bert_finetuning/results_stress/lr_test")
histories_stress_ms_test = load_histories(results_dir="results/bert_finetuning/results_stress/ms_test")
histories_memtype_bs_test = load_histories(results_dir="results/bert_finetuning/results_memtype/bs_test")
histories_memtype_lr_test = load_histories(results_dir="results/bert_finetuning/results_memtype/lr_test")
histories_memtype_ms_test = load_histories(results_dir="results/bert_finetuning/results_memtype/ms_test")
histories_memtype_lr_bs_test_tiny = load_histories(results_dir="results/bert_finetuning/results_memtype/lr_bs_test_tiny")
histories_memtype_lr_bs_test_mini = load_histories(results_dir="results/bert_finetuning/results_memtype/lr_bs_test_mini")

In [ ]:
df_ms = results_table(histories_memtype_ms_test)
df_lr = results_table(histories_memtype_lr_test)
df_bs = results_table(histories_memtype_bs_test)
df_tiny = results_table(histories_memtype_lr_bs_test_tiny)
df_mini = results_table(histories_memtype_lr_bs_test_mini)

display(df_ms)

In [ ]:
metric_map = {
    "Test acc":  "acc",
    "Val acc":   "val_acc_m5",
    "Train acc": "tr_acc_m5",
}
plot_metrics_vs_x(df_ms,   "params_M", metric_map,
                  title="Accuracy vs model size")

plot_metrics_vs_x(df_bs,   "bs",       metric_map,
                  title="Accuracy vs batch size")

plot_metrics_vs_x(df_lr,   "lr",       metric_map,
                  title="Accuracy vs learning-rate")

metric_map = {"Val loss": "val_loss_m5"}
plot_metrics_vs_x(df_ms,      "params_M", metric_map,
                  title="Validation loss vs model size")


In [ ]:

plot_heatmap(df_tiny, index="bs", columns="lr", values="acc",
             title="Accuracy heat-map – 4.4 M-param model")
plot_heatmap(df_mini, index="bs", columns="lr", values="acc",
             title="Accuracy heat-map – 11.3 M-param model")

# 2  focus on validation loss, show 3-digit LRs
plot_heatmap(df_mini, index="bs", columns="lr",
             values="val_loss_m5", cmap="magma",
             fmt_xtick="{:.1e}", title="Val-loss (mean of last 5)")
